In [ ]:
import os, sys, json, subprocess
from kaggle_secrets import UserSecretsClient

s = UserSecretsClient()
TOKEN, HF_TOKEN, PLAIN_URL = s.get_secret("GITHUB_TOKEN"), s.get_secret("HF_TOKEN"), s.get_secret("GITHUB_REPO_URL")
REPO_URL = PLAIN_URL.replace("https://", f"https://{TOKEN}@")
REPO_DIR = "/kaggle/working/qwen2.5-sft-dpo-alignment"

def run(cmd):
    out = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    print((out.stdout + out.stderr).replace(TOKEN, "***"))
    if out.returncode != 0:
        raise RuntimeError("command failed")

def push(msg, notebook=None):
    if notebook:
        import nbformat
        cells = [nbformat.v4.new_code_cell(c) for c in In[1:] if c.strip()]
        nbformat.write(nbformat.v4.new_notebook(cells=cells), f"notebooks/{notebook}")
    run(f'git add results notebooks && (git diff --cached --quiet || git commit -q -m "{msg}") && git push -q {REPO_URL} HEAD:main')

if not os.path.isdir(REPO_DIR):
    run(f"git clone -q {REPO_URL} {REPO_DIR}")
os.chdir(REPO_DIR); sys.path.insert(0, REPO_DIR)
run(f"git remote set-url origin {PLAIN_URL} && git pull -q {REPO_URL} main")
run('git config user.name "kaggle" && git config user.email "kaggle@users.noreply.github.com"')
run('pip install -q "transformers>=4.43.0,<5" accelerate')

from huggingface_hub import login
login(token=HF_TOKEN)

In [ ]:
import random, torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from sft import format_prompt

OUT = "results/task6_generations"
os.makedirs(OUT, exist_ok=True)
GEN = dict(do_sample=False, max_new_tokens=256, repetition_penalty=1.1)        # same as Task 2

repo = lambda path: json.load(open(path))["hf_repo"]
MODELS = {"sft": repo("results/task3_sft/model_location.json"),
          "dpo_uf": repo("results/task4_dpo_uf/model_location.json"),
          "dpo_hh": repo("results/task4_dpo_hh/model_location.json")}
SETS = {"alpaca": json.load(open("eval_sets/alpaca_eval_300_seed42.json")),
        "hh": json.load(open("eval_sets/hh_rlhf_eval_300_seed42.json"))}
JOBS = {"sft": ["alpaca", "hh"], "dpo_uf": ["alpaca"], "dpo_hh": ["hh"]}

@torch.no_grad()
def generate(tok, model, items, name, ds):
    rows = []
    for i in range(0, len(items), 16):
        batch = items[i:i + 16]
        enc = tok([format_prompt(x["instruction"]) for x in batch], return_tensors="pt", padding=True).to("cuda")
        out = model.generate(**enc, **GEN, pad_token_id=tok.pad_token_id)
        texts = tok.batch_decode(out[:, enc["input_ids"].shape[1]:], skip_special_tokens=True)
        rows += [{"id": x["id"], "instruction": x["instruction"], "output": t.split("### Instruction")[0].strip(),
                  "generator": f"qwen2.5-0.5b-{name}", "dataset": ds} for x, t in zip(batch, texts)]
    return rows

for name, sets in JOBS.items():
    tok = AutoTokenizer.from_pretrained(MODELS[name], subfolder="best", padding_side="left")
    model = AutoModelForCausalLM.from_pretrained(MODELS[name], subfolder="best", dtype=torch.float16).cuda().eval()

    # Sanity check: 3 prompts — read them (format OK? stops by itself? sensible length?)
    for r in generate(tok, model, SETS["alpaca"][:2] + SETS["hh"][:1], name, "sanity"):
        print(f"[{name}] Q: {r['instruction'][:90]}\nA ({len(tok(r['output']).input_ids)} tokens): {r['output'][:300]}\n")

    for ds in sets:
        path = f"{OUT}/{name}_{ds}.json"
        if os.path.exists(path):
            print(" already done:", path); continue
        json.dump(generate(tok, model, SETS[ds], name, ds), open(path, "w"), indent=2)
        push(f"Task 6: {name} on {ds}")
        print(" saved + pushed:", path)
    del model, tok; torch.cuda.empty_cache()

In [ ]:
def pairwise(name, dpo_file, sft_file):
    sft = {r["id"]: r for r in json.load(open(sft_file))}
    rng = random.Random(42)
    with open(f"{OUT}/pairwise_{name}.jsonl", "w") as f:
        for d in json.load(open(dpo_file)):
            s = sft[d["id"]]
            a, b = (d, s) if rng.random() < 0.5 else (s, d)              # random order
            f.write(json.dumps({"id": d["id"], "instruction": d["instruction"],
                                "response_a": a["output"], "response_b": b["output"],
                                "hidden": {"model_a": a["generator"], "model_b": b["generator"]}}) + "\n")
    print("✅", f"{OUT}/pairwise_{name}.jsonl")

pairwise("dpo_uf_vs_sft", f"{OUT}/dpo_uf_alpaca.json", f"{OUT}/sft_alpaca.json")
pairwise("dpo_hh_vs_sft", f"{OUT}/dpo_hh_hh.json", f"{OUT}/sft_hh.json")

json.dump({"decoding": GEN, "prompt": "### Instruction:\n{instruction}\n\n### Response:\n",
           "cleanup": "cut at '### Instruction'", "pairwise_seed": 42}, open(f"{OUT}/generation_config.json", "w"), indent=2)
push("Task 6: pairwise files", notebook="25280065_06_generate.ipynb")